# Lightweight Machine Learning for Network Attack Detection on Edge Devices

**Research experiment notebook**

This notebook compares a Random Forest baseline with a shallow Decision Tree using the prepared Edge-IIoTset ML CSV. It evaluates detection quality alongside model size and inference costs. Measurements on this MacBook are host-based results, not proof of performance on a physical edge device. Run one section at a time and inspect the early results before training.

## 1. Experiment plan

- Dataset: `ML-EdgeIIoT-dataset.csv` (confirm the path below).
- Target: `Attack_label`, for binary normal-versus-attack classification.
- Baseline: Random Forest.
- Lightweight candidate: depth-limited Decision Tree.
- Detection metrics: precision, recall, F1, macro F1, false-positive rate, and confusion matrix.
- Resource metrics: model file size, inference latency, CPU time, and process memory readings where available.

Both models use the same stratified train/test split. Preprocessing is fitted on training data only. This is a controlled comparison, not a claim that either model is new.

## 2. Imports and reproducibility

These imports cover inspection, preprocessing, modelling, evaluation, and basic resource measurements. If an import fails, stop and review the message before installing anything.

In [1]:
from pathlib import Path
import os, time, pickle, tempfile
import numpy as np
import pandas as pd
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report)

RANDOM_STATE = 42
TEST_SIZE = 0.20
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("Working directory:", Path.cwd())

pandas: 3.0.2
NumPy: 1.26.4
scikit-learn: 1.5.1
Working directory: /Users/agumekenneth/Documents/year3_sem_1/DS project/Data-Science-Final-year-Project--Prototype-1


## 3. Find the dataset

This checks likely locations without reading the CSV. If it cannot find the file, set `DATASET_PATH` manually to the correct path.

In [2]:
DATASET_FILENAME = "ML-EdgeIIoT-dataset.csv"
roots = [Path.cwd(), *Path.cwd().parents]
candidates = []
for root in roots:
    candidates.extend([
        root / "Edge-IIoTset dataset" / "Selected dataset for ML and DL" / DATASET_FILENAME,
        root / "Data-Science-Final-year-Project--Prototype-1" / "Edge-IIoTset dataset" / "Selected dataset for ML and DL" / DATASET_FILENAME,
    ])
matches = list(dict.fromkeys(p.resolve() for p in candidates if p.is_file()))
DATASET_PATH = matches[0] if matches else None
if DATASET_PATH:
    print("Dataset found:", DATASET_PATH)
    print(f"File size: {DATASET_PATH.stat().st_size / (1024**2):.2f} MB")
    if len(matches) > 1: print("Other copies found:", *matches[1:], sep="\n - ")
else:
    print("Dataset not found automatically. Set DATASET_PATH to its full path.")

Dataset found: /Users/agumekenneth/Documents/year3_sem_1/DS project/Data-Science-Final-year-Project--Prototype-1/Edge-IIoTset dataset/Selected dataset for ML and DL/ML-EdgeIIoT-dataset.csv
File size: 78.38 MB


## 4. Inspect a small sample first

Only 1,000 rows are read here. Check the target column, column names, data types, and missing values before loading the whole file.

In [3]:
if DATASET_PATH is None:
    raise FileNotFoundError("Set DATASET_PATH in Section 3 before continuing.")
sample = pd.read_csv(DATASET_PATH, nrows=1000, low_memory=False)
print("Sample shape:", sample.shape)
print("Columns:", sample.columns.tolist())
display(sample.head())
display(sample.dtypes.astype(str).value_counts().rename_axis("data type").to_frame("column count"))
display(sample.isna().sum().sort_values(ascending=False).head(20).to_frame("missing values"))
if "Attack_label" in sample.columns:
    display(sample["Attack_label"].value_counts(dropna=False).to_frame("sample count"))
else:
    raise KeyError("Attack_label is missing. Verify the target column before proceeding.")

Sample shape: (1000, 63)
Columns: ['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4', 'arp.opcode', 'arp.hw.size', 'arp.src.proto_ipv4', 'icmp.checksum', 'icmp.seq_le', 'icmp.transmit_timestamp', 'icmp.unused', 'http.file_data', 'http.content_length', 'http.request.uri.query', 'http.request.method', 'http.referer', 'http.request.full_uri', 'http.request.version', 'http.response', 'http.tls_port', 'tcp.ack', 'tcp.ack_raw', 'tcp.checksum', 'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.dstport', 'tcp.flags', 'tcp.flags.ack', 'tcp.len', 'tcp.options', 'tcp.payload', 'tcp.seq', 'tcp.srcport', 'udp.port', 'udp.stream', 'udp.time_delta', 'dns.qry.name', 'dns.qry.name.len', 'dns.qry.qu', 'dns.qry.type', 'dns.retransmission', 'dns.retransmit_request', 'dns.retransmit_request_in', 'mqtt.conack.flags', 'mqtt.conflag.cleansess', 'mqtt.conflags', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.msg_decoded_as', 'mqtt.msg', 'mqtt.msgtype', 'mqtt.pro

,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
1,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
2,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
3,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
4,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM


,column count
data type,
float64,58
str,3
int64,2


,missing values
frame.time,0
mqtt.conflags,0
tcp.srcport,0
udp.port,0
udp.stream,0
udp.time_delta,0
dns.qry.name,0
dns.qry.name.len,0
dns.qry.qu,0
dns.qry.type,0


,sample count
Attack_label,
1,1000


## 5. Load the full CSV and inspect class balance

The prepared file was reported as approximately 78 MB, but its in-memory size will be larger. This reads the full dataset for the experiment.

In [4]:
df = pd.read_csv(DATASET_PATH, low_memory=False)
print("Full dataset shape:", df.shape)
print(f"Approximate DataFrame memory: {df.memory_usage(deep=True).sum() / (1024**2):.1f} MB")
if "Attack_label" not in df.columns: raise KeyError("Attack_label is missing from the full dataset.")
display(df["Attack_label"].value_counts(dropna=False).sort_index().to_frame("count"))
display(df["Attack_label"].value_counts(normalize=True, dropna=False).sort_index().to_frame("proportion"))

Full dataset shape: (157800, 63)
Approximate DataFrame memory: 123.1 MB


,count
Attack_label,
0,24301
1,133499


,proportion
Attack_label,
0,0.153999
1,0.846001


## 6. Prepare features and target

`Attack_label` is the target. We drop `Attack_type` because it describes the attack family and could reveal the answer, and drop `frame.time` for this first binary experiment. Rows without a target are removed. Unexpected target values cause the notebook to stop rather than guess.

In [5]:
model_df = df.dropna(subset=["Attack_label"]).copy()
raw_target = model_df["Attack_label"].astype(str).str.strip()
target_map = {"0": 0, "0.0": 0, "Normal": 0, "normal": 0, "1": 1, "1.0": 1, "Attack": 1, "attack": 1}
y = raw_target.map(target_map)
if y.isna().any():
    raise ValueError("Unexpected Attack_label values; inspect them before modelling:\n" + str(raw_target[y.isna()].value_counts().head(20)))
y = y.astype("int8")
X = model_df.drop(columns=["Attack_label", "Attack_type", "frame.time"], errors="ignore")
empty_columns = X.columns[X.isna().all()].tolist()
if empty_columns:
    print("Dropping completely empty columns:", empty_columns)
    X = X.drop(columns=empty_columns)
numeric_cols = X.select_dtypes(include=["number", "bool"]).columns
X.loc[:, numeric_cols] = X.loc[:, numeric_cols].replace([np.inf, -np.inf], np.nan)
print("Feature matrix:", X.shape)
print("Categorical columns:", X.select_dtypes(exclude=["number", "bool"]).columns.tolist())
display(y.value_counts().sort_index().rename(index={0:"Normal (0)", 1:"Attack (1)"}).to_frame("count"))

Feature matrix: (157800, 60)
Categorical columns: ['ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4', 'arp.src.proto_ipv4', 'http.file_data', 'http.request.uri.query', 'http.request.method', 'http.referer', 'http.request.full_uri', 'http.request.version', 'tcp.options', 'tcp.payload', 'tcp.srcport', 'dns.qry.name.len', 'mqtt.conack.flags', 'mqtt.msg', 'mqtt.protoname', 'mqtt.topic']


,count
Attack_label,
Normal (0),24301
Attack (1),133499


## 7. Split first, then fit preprocessing

A stratified split keeps class proportions similar between training and test data. Imputers and the encoder live inside the pipeline, so they are fitted only on training data.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number", "bool"]).columns.tolist()
preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric_features),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
], remainder="drop")
print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Training rows: 126240 | Test rows: 31560
Numeric features: 42
Categorical features: 18


## 8. Define and train the models

Random Forest is the baseline. The shallow Decision Tree is the lightweight candidate; limiting its depth constrains model complexity. These are starting settings, not necessarily optimal settings. Do not tune using the held-out test set.

In [7]:
models = {
    "Random Forest (baseline)": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced"),
    "Shallow Decision Tree (lightweight)": DecisionTreeClassifier(max_depth=6, min_samples_leaf=5, random_state=RANDOM_STATE, class_weight="balanced"),
}
def evaluate_binary(y_true, y_pred):
    # Fixed label order keeps the confusion-matrix positions clear.
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {"accuracy": accuracy_score(y_true, y_pred), "attack_precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
            "attack_recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0), "attack_f1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
            "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0), "false_positive_rate": fp / (fp + tn) if (fp + tn) else np.nan,
            "TN": tn, "FP": fp, "FN": fn, "TP": tp}
fitted_models, predictions, metric_rows = {}, {}, []
for name, estimator in models.items():
    print("Training", name)
    pipeline = Pipeline([("preprocess", preprocessor), ("model", estimator)])
    start = time.perf_counter()
    pipeline.fit(X_train, y_train)
    training_seconds = time.perf_counter() - start
    y_pred = pipeline.predict(X_test)
    row = evaluate_binary(y_test, y_pred)
    row["model"] = name
    row["training_seconds"] = training_seconds
    fitted_models[name], predictions[name] = pipeline, y_pred
    metric_rows.append(row)
    print(f"Training time: {training_seconds:.2f} seconds")
    print(classification_report(y_test, y_pred, labels=[0, 1], target_names=["Normal", "Attack"], zero_division=0))
metrics_df = pd.DataFrame(metric_rows).set_index("model")
display(metrics_df.round(4))

Training Random Forest (baseline)
Training time: 17.86 seconds
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00      4860
      Attack       1.00      1.00      1.00     26700

    accuracy                           1.00     31560
   macro avg       1.00      1.00      1.00     31560
weighted avg       1.00      1.00      1.00     31560

Training Shallow Decision Tree (lightweight)
Training time: 47.04 seconds
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00      4860
      Attack       1.00      1.00      1.00     26700

    accuracy                           1.00     31560
   macro avg       1.00      1.00      1.00     31560
weighted avg       1.00      1.00      1.00     31560



,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,false_positive_rate,TN,FP,FN,TP,training_seconds
model,,,,,,,,,,,
Random Forest (baseline),1.0,1.0,1.0,1.0,1.0,0.0,4860,0,0,26700,17.8623
Shallow Decision Tree (lightweight),1.0,1.0,1.0,1.0,1.0,0.0,4860,0,0,26700,47.0441


## 9. Confusion matrices

False negatives are attacks that were missed. False positives are normal records flagged as attacks. Both matter when judging whether a detector is useful.

In [8]:
for name, y_pred in predictions.items():
    print(name)
    display(pd.DataFrame(confusion_matrix(y_test, y_pred, labels=[0, 1]), index=["Actual normal", "Actual attack"], columns=["Predicted normal", "Predicted attack"]))

Random Forest (baseline)


,Predicted normal,Predicted attack
Actual normal,4860,0
Actual attack,0,26700


Shallow Decision Tree (lightweight)


,Predicted normal,Predicted attack
Actual normal,4860,0
Actual attack,0,26700


## 10. Resource measurements on this host

We measure inference on the same held-out subset for both models. The notebook records serialized model size, total inference time, mean and 95th-percentile batch latency, and CPU/RSS readings if `psutil` is already installed. RSS before/after is **not** a precise peak-memory measurement. These are host-based measures, not physical edge-device results. Models are written only to a temporary directory.

In [9]:
try:
    import psutil
    HAS_PSUTIL = True
except ImportError:
    HAS_PSUTIL = False
    print("psutil is unavailable; CPU and RSS readings will be omitted. No packages will be installed.")
proc = psutil.Process(os.getpid()) if HAS_PSUTIL else None
X_inference = X_test.iloc[:min(5000, len(X_test))]
resource_rows = []
with tempfile.TemporaryDirectory(prefix="nids_model_measurements_") as temp_dir:
    for name, pipeline in fitted_models.items():
        filename = name.lower().replace(" ", "_").replace("(", "").replace(")", "") + ".pkl"
        model_path = Path(temp_dir) / filename
        with model_path.open("wb") as f: pickle.dump(pipeline, f)
        model_size_mb = model_path.stat().st_size / (1024**2)
        cpu_before = proc.cpu_times() if HAS_PSUTIL else None
        rss_before = proc.memory_info().rss / (1024**2) if HAS_PSUTIL else np.nan
        # Warm up the model so first-call overhead is less likely to dominate.
        pipeline.predict(X_inference.iloc[:min(100, len(X_inference))])
        latencies = []
        for start in range(0, len(X_inference), 250):
            batch = X_inference.iloc[start:start + 250]
            t0 = time.perf_counter()
            pipeline.predict(batch)
            latencies.append((time.perf_counter() - t0) * 1000)
        cpu_after = proc.cpu_times() if HAS_PSUTIL else None
        rss_after = proc.memory_info().rss / (1024**2) if HAS_PSUTIL else np.nan
        cpu_seconds = ((cpu_after.user + cpu_after.system) - (cpu_before.user + cpu_before.system)) if HAS_PSUTIL else np.nan
        resource_rows.append({"model": name, "model_size_MB": model_size_mb, "inference_rows": len(X_inference),
            "total_inference_seconds": sum(latencies) / 1000, "mean_batch_latency_ms": float(np.mean(latencies)),
            "p95_batch_latency_ms": float(np.percentile(latencies, 95)), "CPU_seconds_during_timing": cpu_seconds,
            "RSS_before_inference_MB": rss_before, "RSS_after_inference_MB": rss_after})
resource_df = pd.DataFrame(resource_rows).set_index("model")
display(resource_df.round(4))

,model_size_MB,inference_rows,total_inference_seconds,mean_batch_latency_ms,p95_batch_latency_ms,CPU_seconds_during_timing,RSS_before_inference_MB,RSS_after_inference_MB
model,,,,,,,,
Random Forest (baseline),24.6880,5000,11.8949,594.7433,880.0851,10.2162,740.3398,713.418
Shallow Decision Tree (lightweight),23.1095,5000,10.3777,518.8835,809.3650,8.9144,717.3281,712.500


## 11. Compare detection and resource results

Review the detection metrics and resource measurements together. A smaller model is not automatically better if it misses substantially more attacks.

In [10]:
comparison_df = metrics_df.join(resource_df, how="left")
display(comparison_df.round(4))

,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,false_positive_rate,TN,FP,FN,TP,training_seconds,model_size_MB,inference_rows,total_inference_seconds,mean_batch_latency_ms,p95_batch_latency_ms,CPU_seconds_during_timing,RSS_before_inference_MB,RSS_after_inference_MB
model,,,,,,,,,,,,,,,,,,,
Random Forest (baseline),1.0,1.0,1.0,1.0,1.0,0.0,4860,0,0,26700,17.8623,24.6880,5000,11.8949,594.7433,880.0851,10.2162,740.3398,713.418
Shallow Decision Tree (lightweight),1.0,1.0,1.0,1.0,1.0,0.0,4860,0,0,26700,47.0441,23.1095,5000,10.3777,518.8835,809.3650,8.9144,717.3281,712.500


## 12. Discussion guide

Use the actual results to answer these questions:

1. How did attack recall and macro F1 change between the models?
2. Did the shallow tree reduce model size or inference latency?
3. What happened to the false-positive rate?
4. Is the trade-off acceptable for the detection task?
5. Which resource measures were unavailable or limited?
6. What would need to be measured again on a real constrained edge device?

Do not insert expected or invented numbers into the report.

## 13. Limitations and next steps

- This first experiment uses one dataset and one fixed train/test split.
- A random split may overestimate generalization if duplicate or closely related records appear in both sets. Review duplicates and consider a split that reflects the intended deployment setting.
- `Attack_type` and `frame.time` are excluded, but other fields should still be reviewed for identifiers or leakage.
- MacBook measurements do not establish performance on a physical edge device.
- RSS readings are before/after snapshots, not precise peak-memory measurements.
- Any tuning should use training data or cross-validation, not the held-out test set.
- If classes are imbalanced, discuss class-specific metrics rather than relying on accuracy alone.

**Next step:** run the notebook section by section, inspect outputs, and document settings and software versions before adding multiclass classification or another dataset.